# Experiment 2: Create Backend for Model Inference (FastAPI)

**Objective:**
- Create a FastAPI backend for serving the trained ML model
- Define a `/predict` endpoint for model inference
- Use Pydantic for request & response schemas

**Prerequisites:** Run Experiment 1 first to generate the model artifacts.

## Step 1: Install Required Libraries

In [ ]:
# Install FastAPI and dependencies
# - FastAPI: modern web framework for building APIs (faster than Flask, auto docs)
# - uvicorn: ASGI server to run FastAPI apps
# - pydantic: data validation using Python type hints
# - nest_asyncio: allows running asyncio in Jupyter notebooks
# - requests: test API endpoints from within notebook
import sys
!{sys.executable} -m pip install fastapi uvicorn pydantic scikit-learn pandas numpy nest-asyncio requests

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Step 2: Import Libraries

In [ ]:
# Import libraries for API development and testing
# - FastAPI: create REST API endpoints
# - Pydantic: define request/response schemas with validation
# - Literal: restrict field values to specific options (e.g., Gender: "Male" or "Female")
# - threading/uvicorn: run API server in background within notebook
# - nest_asyncio.apply(): allows async operations in Jupyter
import pickle
import numpy as np
import pandas as pd
import uvicorn
import nest_asyncio
import threading
import time
import requests

from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Optional, Literal

nest_asyncio.apply()
print("All libraries imported successfully!")

All libraries imported successfully!


## Step 3: Load Model Artifacts

In [ ]:
# Load model artifacts from Experiment 1
# - Load ONCE at startup (not per request) for efficiency
# - All 4 artifacts required: model, scaler, encoders, feature_names
# - If any artifact is missing, API will fail to start (fail-fast principle)
# - In production, these would be loaded from model registry or cloud storage
# Load all model artifacts
with open('model_artifacts/churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
print(f"Model loaded: {type(model).__name__}")

with open('model_artifacts/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print(f"Scaler loaded: {type(scaler).__name__}")

with open('model_artifacts/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)
print(f"Label encoders loaded: {list(label_encoders.keys())}")

with open('model_artifacts/feature_names.pkl', 'rb') as f:
    feature_names = pickle.load(f)
print(f"Feature names: {feature_names}")

Model loaded: RandomForestClassifier
Scaler loaded: StandardScaler
Label encoders loaded: ['Gender', 'Contract', 'PaymentMethod']
Feature names: ['Gender', 'SeniorCitizen', 'Tenure', 'MonthlyCharges', 'Contract', 'PaymentMethod', 'TotalCharges']


## Step 4: Define Pydantic Request & Response Schemas

In [ ]:
# Define Pydantic schemas for request validation and response structure
# - ChurnPredictionRequest: validates incoming JSON (types, ranges, allowed values)
# - Literal['Male', 'Female']: only these values allowed (rejects invalid inputs)
# - Field(..., ge=0, le=1): validates SeniorCitizen is 0 or 1
# - Field(..., ge=0): ensures non-negative values for charges
# - ChurnPredictionResponse: standardizes API output format
# - json_schema_extra: provides example payloads in auto-generated Swagger docs
# Request schema
class ChurnPredictionRequest(BaseModel):
    """Schema for churn prediction request."""
    Gender: Literal['Male', 'Female'] = Field(..., description="Customer gender")
    SeniorCitizen: int = Field(..., ge=0, le=1, description="Whether customer is senior citizen (0 or 1)")
    Tenure: int = Field(..., ge=0, description="Number of months the customer has stayed")
    MonthlyCharges: float = Field(..., ge=0, description="Monthly charges in dollars")
    Contract: Literal['Month-to-month', 'One year', 'Two year'] = Field(..., description="Contract type")
    PaymentMethod: Literal['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'] = Field(..., description="Payment method")
    TotalCharges: float = Field(..., ge=0, description="Total charges in dollars")

    class Config:
        json_schema_extra = {
            "example": {
                "Gender": "Male",
                "SeniorCitizen": 0,
                "Tenure": 30,
                "MonthlyCharges": 70.5,
                "Contract": "One year",
                "PaymentMethod": "Electronic check",
                "TotalCharges": 2115.0
            }
        }

# Response schema
class ChurnPredictionResponse(BaseModel):
    """Schema for churn prediction response."""
    prediction: int = Field(..., description="Predicted class (0: No Churn, 1: Churn)")
    prediction_label: str = Field(..., description="Human-readable prediction label")
    churn_probability: float = Field(..., description="Probability of churn")
    no_churn_probability: float = Field(..., description="Probability of no churn")

    class Config:
        json_schema_extra = {
            "example": {
                "prediction": 0,
                "prediction_label": "No Churn",
                "churn_probability": 0.15,
                "no_churn_probability": 0.85
            }
        }

# Health check response
class HealthResponse(BaseModel):
    status: str
    model_loaded: bool

print("Pydantic schemas defined successfully!")
print(f"\nRequest schema fields: {list(ChurnPredictionRequest.model_fields.keys())}")
print(f"Response schema fields: {list(ChurnPredictionResponse.model_fields.keys())}")

Pydantic schemas defined successfully!

Request schema fields: ['Gender', 'SeniorCitizen', 'Tenure', 'MonthlyCharges', 'Contract', 'PaymentMethod', 'TotalCharges']
Response schema fields: ['prediction', 'prediction_label', 'churn_probability', 'no_churn_probability']


/Users/harshsmac/Library/Python/3.9/lib/python/site-packages/pydantic/_internal/_fields.py:149: UserWarning: Field "model_loaded" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## Step 5: Create the FastAPI Application

In [ ]:
# Create FastAPI application with 3 endpoints
# - GET /: root endpoint, returns API info (basic health check)
# - GET /health: detailed health check (verifies model is loaded)
# - POST /predict: main inference endpoint
#   1. Receives ChurnPredictionRequest (auto-validated by Pydantic)
#   2. Converts to DataFrame for preprocessing
#   3. Applies label encoding (same as training)
#   4. Applies scaling (same as training)
#   5. Runs model.predict() and model.predict_proba()
#   6. Returns structured ChurnPredictionResponse
# - tags=["General"]: groups endpoints in Swagger UI
# Create FastAPI app
app = FastAPI(
    title="Bank Churn Prediction API",
    description="API for predicting customer churn using a trained ML model",
    version="1.0.0"
)

@app.get("/", tags=["General"])
def root():
    """Root endpoint."""
    return {"message": "Bank Churn Prediction API", "version": "1.0.0"}

@app.get("/health", response_model=HealthResponse, tags=["General"])
def health_check():
    """Health check endpoint."""
    return HealthResponse(status="healthy", model_loaded=model is not None)

@app.post("/predict", response_model=ChurnPredictionResponse, tags=["Prediction"])
def predict_churn(request: ChurnPredictionRequest):
    """Predict customer churn based on input features."""
    # Convert request to DataFrame
    input_data = pd.DataFrame([request.model_dump()])
    
    # Encode categorical variables
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    
    # Scale features
    input_scaled = scaler.transform(input_data)
    
    # Make prediction
    prediction = model.predict(input_scaled)[0]
    probabilities = model.predict_proba(input_scaled)[0]
    
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probabilities[1]), 4),
        no_churn_probability=round(float(probabilities[0]), 4)
    )

print("FastAPI application created with endpoints:")
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f"  {list(route.methods)} {route.path}")

FastAPI application created with endpoints:
  ['GET', 'HEAD'] /openapi.json
  ['GET', 'HEAD'] /docs
  ['GET', 'HEAD'] /docs/oauth2-redirect
  ['GET', 'HEAD'] /redoc
  ['GET'] /
  ['GET'] /health
  ['POST'] /predict


## Step 6: Write the FastAPI App to a Python File

This creates a standalone `app.py` file that can be run independently.

In [ ]:
# Export FastAPI app to standalone Python file (app.py)
# - This is the PRODUCTION deployment artifact
# - Can be run with: uvicorn app:app --reload
# - Loads model artifacts at import time (not per request)
# - Contains same endpoints as notebook version
# - Why separate file? Enables deployment via Docker, cloud services, systemd, etc.
# - In production, you'd deploy this app.py, not the notebook
app_code = '''import pickle
import numpy as np
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal

# Load model artifacts
with open("model_artifacts/churn_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("model_artifacts/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("model_artifacts/label_encoders.pkl", "rb") as f:
    label_encoders = pickle.load(f)
with open("model_artifacts/feature_names.pkl", "rb") as f:
    feature_names = pickle.load(f)

# Pydantic Schemas
class ChurnPredictionRequest(BaseModel):
    Gender: Literal["Male", "Female"] = Field(..., description="Customer gender")
    SeniorCitizen: int = Field(..., ge=0, le=1, description="Senior citizen (0 or 1)")
    Tenure: int = Field(..., ge=0, description="Months of tenure")
    MonthlyCharges: float = Field(..., ge=0, description="Monthly charges")
    Contract: Literal["Month-to-month", "One year", "Two year"] = Field(..., description="Contract type")
    PaymentMethod: Literal["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"] = Field(..., description="Payment method")
    TotalCharges: float = Field(..., ge=0, description="Total charges")

    class Config:
        json_schema_extra = {
            "example": {
                "Gender": "Male", "SeniorCitizen": 0, "Tenure": 30,
                "MonthlyCharges": 70.5, "Contract": "One year",
                "PaymentMethod": "Electronic check", "TotalCharges": 2115.0
            }
        }

class ChurnPredictionResponse(BaseModel):
    prediction: int = Field(..., description="Predicted class (0 or 1)")
    prediction_label: str = Field(..., description="Human-readable label")
    churn_probability: float = Field(..., description="Probability of churn")
    no_churn_probability: float = Field(..., description="Probability of no churn")

class HealthResponse(BaseModel):
    status: str
    model_loaded: bool

# FastAPI App
app = FastAPI(
    title="Bank Churn Prediction API",
    description="API for predicting customer churn",
    version="1.0.0"
)

@app.get("/")
def root():
    return {"message": "Bank Churn Prediction API", "version": "1.0.0"}

@app.get("/health", response_model=HealthResponse)
def health_check():
    return HealthResponse(status="healthy", model_loaded=model is not None)

@app.post("/predict", response_model=ChurnPredictionResponse)
def predict_churn(request: ChurnPredictionRequest):
    input_data = pd.DataFrame([request.model_dump()])
    for col in label_encoders:
        if col in input_data.columns:
            input_data[col] = label_encoders[col].transform(input_data[col])
    input_scaled = scaler.transform(input_data)
    prediction = model.predict(input_scaled)[0]
    probabilities = model.predict_proba(input_scaled)[0]
    return ChurnPredictionResponse(
        prediction=int(prediction),
        prediction_label="Churn" if prediction == 1 else "No Churn",
        churn_probability=round(float(probabilities[1]), 4),
        no_churn_probability=round(float(probabilities[0]), 4)
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("FastAPI app written to app.py")
print("Run with: uvicorn app:app --reload --host 0.0.0.0 --port=8000")

FastAPI app written to app.py
Run with: uvicorn app:app --reload --host 0.0.0.0 --port 8000


## Step 7: Run and Test the API Server

In [ ]:
# Start API server in background thread (for testing within notebook)
# - threading.Thread: runs server without blocking notebook execution
# - daemon=True: thread terminates when notebook kernel stops
# - time.sleep(3): wait for server initialization
# - In production, run with: uvicorn app:app --workers 4 --host 0.0.0.0 --port 8000
# - Access Swagger UI at: http://localhost:8000/docs
# Start the server in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # Wait for server to start
print("Server started on http://localhost:8000")
print("API docs available at http://localhost:8000/docs")

INFO:     Started server process [15571]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Server started on http://localhost:8000
API docs available at http://localhost:8000/docs


INFO:     127.0.0.1:55636 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:55638 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:55640 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:55642 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:55649 - "POST /predict HTTP/1.1" 200 OK
INFO:     127.0.0.1:55651 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:55653 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:55655 - "POST /predict HTTP/1.1" 422 Unprocessable Entity
INFO:     127.0.0.1:55657 - "GET /test-error HTTP/1.1" 404 Not Found


In [ ]:
# Test root and health endpoints
# - GET /: basic info endpoint (no business logic, quick sanity check)
# - GET /health: verifies model is loaded and API is ready
# - Status 200: success
# - In production, health endpoint used by load balancers and Kubernetes probes
# Test the root endpoint
response = requests.get("http://localhost:8000/")
print(f"Root Endpoint - Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test the health endpoint
response = requests.get("http://localhost:8000/health")
print(f"\nHealth Endpoint - Status: {response.status_code}")
print(f"Response: {response.json()}")

Root Endpoint - Status: 200
Response: {'message': 'Bank Churn Prediction API', 'version': '1.0.0'}

Health Endpoint - Status: 200
Response: {'status': 'healthy', 'model_loaded': True}


In [ ]:
# Test predict endpoint with two scenarios
# - Test 1: Low-risk customer (long tenure, yearly contract, stable payment)
# - Test 2: High-risk customer (short tenure, month-to-month, electronic check)
# - Validates end-to-end pipeline: JSON → validation → encoding → scaling → prediction → response
# - Check predictions make business sense (high tenure = less churn)
# Test the /predict endpoint
test_payload = {
    "Gender": "Male",
    "SeniorCitizen": 0,
    "Tenure": 30,
    "MonthlyCharges": 70.5,
    "Contract": "One year",
    "PaymentMethod": "Electronic check",
    "TotalCharges": 2115.0
}

response = requests.post("http://localhost:8000/predict", json=test_payload)
print(f"Predict Endpoint - Status: {response.status_code}")
print(f"Response: {response.json()}")

# Test with a high-risk customer
test_payload_2 = {
    "Gender": "Female",
    "SeniorCitizen": 1,
    "Tenure": 2,
    "MonthlyCharges": 95.0,
    "Contract": "Month-to-month",
    "PaymentMethod": "Electronic check",
    "TotalCharges": 190.0
}

response = requests.post("http://localhost:8000/predict", json=test_payload_2)
print(f"\nHigh-risk customer prediction - Status: {response.status_code}")
print(f"Response: {response.json()}")

print("\n✅ FastAPI backend is working correctly!")

Predict Endpoint - Status: 200
Response: {'prediction': 0, 'prediction_label': 'No Churn', 'churn_probability': 0.2852, 'no_churn_probability': 0.7148}

High-risk customer prediction - Status: 200
Response: {'prediction': 0, 'prediction_label': 'No Churn', 'churn_probability': 0.3254, 'no_churn_probability': 0.6746}

✅ FastAPI backend is working correctly!
